## Is there a significant difference between political preferences of the voters depending on their demographic attributes recorded in the survey (that is, age, gender, education level…)?

In [7]:
import pandas as pd, numpy as np
from scipy import stats

survey = pd.read_excel("../data/private_dataF.xlsx")

In [8]:
# prepocessing

survey = survey[survey["party"] != "Invalid vote"]
survey["party"].value_counts()

party
Green    127
Red       67
Name: count, dtype: int64

## On Age - at date of data collection 1/7/2025

In [9]:
import numpy as np
import pandas as pd
from datetime import date

date_then = date(2025, 7, 1)

survey_w_age = survey.copy()

survey_w_age["age"] = survey_w_age["dob"].apply(
    lambda dob: date_then.year - dob.year
    - ((date_then.month, date_then.day) < (dob.month, dob.day))
)

survey_w_age["age_group"] = pd.cut(
    survey_w_age["age"],
    bins=[18, 30, 45, 65, np.inf],
    labels=["18-29", "30-44", "45-64", "65 or older"],
    right=False,
)

survey_w_age


,name,sex,evote,dob,zip,education,citizenship,marital_status,party,age,age_group
0,"Casas-Marta, Angelica",Female,0,1997-05-24,2100,Upper secondary education,Denmark,Never married,Green,28,18-29
1,"el-Salahuddin, Rawda",Female,0,1964-01-03,2300,Vocational Education and Training (VET),Denmark,Never married,Red,61,45-64
2,"al-Haider, Saahira",Female,1,1976-01-15,2400,Vocational bachelors educations,Denmark,Never married,Green,49,45-64
3,"Hansen, Karan",Male,1,1960-04-14,2400,Masters programmes,Denmark,Never married,Red,65,65 or older
4,"Lee, Cody",Male,0,1984-10-17,2100,Vocational Education and Training (VET),Denmark,Never married,Green,40,30-44
...,...,...,...,...,...,...,...,...,...,...,...
194,"Graham, Tin",Male,0,1964-01-06,2300,Vocational Education and Training (VET),Denmark,Divorced,Red,61,45-64
195,"Rutherford, Joshua",Male,0,1940-05-15,2400,Vocational Education and Training (VET),Denmark,Married/separated,Green,85,65 or older
196,"Santee, Michael",Male,1,1964-08-22,2400,Vocational Education and Training (VET),Denmark,Married/separated,Green,60,45-64
197,"Elwood, Adrienne",Female,0,1966-04-24,2400,Vocational Education and Training (VET),Denmark,Married/separated,Green,59,45-64


H0: The distribution of political preference is the same across age groups.
HA: The distribution of political preference is the NOT the same across age groups.

chi-square test compares these observed counts with the counts we would expect if age and party were independent.

In [10]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    survey_w_age["age_group"],
    survey_w_age["party"]
)

print(table)
chi2, p, dof, expected = chi2_contingency(table)

print(f"Chi2 coeff: {chi2}, p: {p}, degrees_of_freedom: {dof}")
print(f"expected frequency: {expected}")

party        Green  Red
age_group              
18-29           25    2
30-44           32   15
45-64           47   33
65 or older     23   17
Chi2 coeff: 11.649100261985616, p: 0.00868715484496827, degrees_of_freedom: 3
expected frequency: [[17.67525773  9.32474227]
 [30.76804124 16.23195876]
 [52.37113402 27.62886598]
 [26.18556701 13.81443299]]


A chi-square test of independence found a statistically significant association 
between age group and party preference, $\chi^2(3) = 11.65$, $p = 0.009$. 
All expected cell counts exceeded 5. Respondents aged 18-29 were more likely 
than expected to prefer the Green party and less likely than expected to prefer 
the Red party.

## On Gender

H0: The distribution of political preference is the same for female and male voters.
HA: The distribution of political preference is NOT the same for female and male voters.

chi-square test compares the observed counts with the counts we would expect if gender and party were independent. Since gender has only two categories (2x2 table), scipy applies Yates' continuity correction by default.

In [11]:
table_gender = pd.crosstab(
    survey["sex"],
    survey["party"]
)

print(table_gender)
chi2, p, dof, expected = chi2_contingency(table_gender)

print(f"Chi2 coeff: {chi2}, p: {p}, degrees_of_freedom: {dof}")
print(f"expected frequency: {expected}")

party   Green  Red
sex               
Female     57   39
Male       70   28
Chi2 coeff: 2.6060539990470257, p: 0.10645635974774215, degrees_of_freedom: 1
expected frequency: [[62.84536082 33.15463918]
 [64.15463918 33.84536082]]


A chi-square test of independence (with Yates' continuity correction) found no statistically significant association 
between gender and party preference, $\chi^2(1) = 2.61$, $p = 0.106$. 
All expected cell counts exceeded 5. Male respondents (70 Green, 28 Red) were somewhat more likely than expected to prefer the Green party 
than female respondents (57 Green, 39 Red), but the difference is not significant at the 5% level, 
so we fail to reject H0. Without the correction the result is also not significant ($\chi^2(1) = 3.12$, $p = 0.078$).

## On Education

Education levels are grouped into Low / Medium / High (same mapping as in part C), and respondents with "Not stated" education are removed.

H0: The distribution of political preference is the same across education levels.
HA: The distribution of political preference is NOT the same across education levels.

chi-square test compares the observed counts with the counts we would expect if education and party were independent.

In [12]:
survey_w_edu = survey[survey["education"] != "Not stated"].copy()

education_mapping = {
    "Primary education": "Low",
    "Upper secondary education": "Medium",
    "Vocational Education and Training (VET)": "Medium",
    "Short cycle higher education": "High",
    "Vocational bachelors educations": "High",
    "Bachelors programmes": "High",
    "Masters programmes": "High",
    "PhD programmes": "High"
}

survey_w_edu["education"] = survey_w_edu["education"].map(education_mapping)

print(survey_w_edu["education"].value_counts(dropna=False))

table_edu = pd.crosstab(
    survey_w_edu["education"],
    survey_w_edu["party"]
)

print(table_edu)
chi2, p, dof, expected = chi2_contingency(table_edu)

print(f"Chi2 coeff: {chi2}, p: {p}, degrees_of_freedom: {dof}")
print(f"expected frequency: {expected}")

education
Medium    82
High      71
Low       37
Name: count, dtype: int64
party      Green  Red
education            
High          29   42
Low           29    8
Medium        65   17
Chi2 coeff: 28.354662456211923, p: 6.964072432567518e-07, degrees_of_freedom: 2
expected frequency: [[45.96315789 25.03684211]
 [23.95263158 13.04736842]
 [53.08421053 28.91578947]]


A chi-square test of independence found a statistically significant association 
between education level and party preference, $\chi^2(2) = 28.35$, $p < 0.001$. 
All expected cell counts exceeded 5. Respondents with Low (29 Green, 8 Red) and Medium (65 Green, 17 Red) education 
were more likely than expected to prefer the Green party, while respondents with High education (29 Green, 42 Red) 
were more likely than expected to prefer the Red party. We reject H0.

## Effect size - Cramér's V

p-values only say *whether* there is an association, not *how strong* it is. Cramér's V (0 = no association, 1 = perfect association) is computed from the chi-square statistic as $V = \sqrt{\chi^2 / (n \cdot (\min(r, c) - 1))}$. The uncorrected chi-square statistic is used so the value is comparable across tests.

In [13]:
def cramers_v(table):
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.values.sum()
    return np.sqrt(chi2 / (n * (min(table.shape) - 1)))

print("age       V =", round(cramers_v(table), 3))
print("gender    V =", round(cramers_v(table_gender), 3))
print("education V =", round(cramers_v(table_edu), 3))

age       V = 0.245
gender    V = 0.127
education V = 0.386


## On Citizenship - Danish vs. non-Danish

Citizenship has many countries with only 1-3 respondents each, so it is grouped into Danish vs. non-Danish.

H0: The distribution of political preference is the same for Danish and non-Danish citizens.
HA: The distribution of political preference is NOT the same for Danish and non-Danish citizens.

In [14]:
survey_w_cit = survey.copy()
survey_w_cit["citizenship"] = survey_w_cit["citizenship"].apply(lambda x: "Danish" if x == "Denmark" else "non-danish")

table_cit = pd.crosstab(
    survey_w_cit["citizenship"],
    survey_w_cit["party"]
)

print(table_cit)
chi2, p, dof, expected = chi2_contingency(table_cit)

print(f"Chi2 coeff: {chi2}, p: {p}, degrees_of_freedom: {dof}")
print(f"expected frequency: {expected}")
print("Cramer's V =", round(cramers_v(table_cit), 3))

# small group (22 non-Danish) so we double check with Fisher's exact test
odds, p_fisher = stats.fisher_exact(table_cit)
print(f"Fisher exact: odds ratio: {odds}, p: {p_fisher}")

party        Green  Red
citizenship            
Danish         114   58
non-danish      13    9
Chi2 coeff: 0.18452200965176294, p: 0.6675155564611044, degrees_of_freedom: 1
expected frequency: [[112.59793814  59.40206186]
 [ 14.40206186   7.59793814]]
Cramer's V = 0.048
Fisher exact: odds ratio: 1.3607427055702919, p: 0.4873683938215236


A chi-square test of independence found no statistically significant association 
between citizenship (Danish vs. non-Danish) and party preference, $\chi^2(1) = 0.18$, $p = 0.668$, Cramér's V $= 0.05$ (negligible). 
All expected cell counts exceeded 5. Because the non-Danish group is small (22 respondents), Fisher's exact test was also run and agrees ($p = 0.487$). We fail to reject H0.

## On Zip code (as a category)

Zip is a postal code, not a quantity, so it is treated as a categorical variable.

H0: The distribution of political preference is the same across zip codes.
HA: The distribution of political preference is NOT the same across zip codes.

In [15]:
survey_w_zip = survey.copy()
survey_w_zip["zip"] = survey_w_zip["zip"].astype(str)

table_zip = pd.crosstab(
    survey_w_zip["zip"],
    survey_w_zip["party"]
)

print(table_zip)
chi2, p, dof, expected = chi2_contingency(table_zip)

print(f"Chi2 coeff: {chi2}, p: {p}, degrees_of_freedom: {dof}")
print(f"expected frequency: {expected}")
print("Cramer's V =", round(cramers_v(table_zip), 3))

party  Green  Red
zip              
2100      32    7
2200      31   17
2300      17   17
2400      47   26
Chi2 coeff: 8.396522620339223, p: 0.038489657792234976, degrees_of_freedom: 3
expected frequency: [[25.53092784 13.46907216]
 [31.42268041 16.57731959]
 [22.25773196 11.74226804]
 [47.78865979 25.21134021]]
Cramer's V = 0.208


A chi-square test of independence found a statistically significant association 
between zip code and party preference, $\chi^2(3) = 8.40$, $p = 0.038$, Cramér's V $= 0.21$ (small-to-medium). 
All expected cell counts exceeded 5. Zip 2100 is the most Green-leaning (32 Green vs. 7 Red), while zip 2300 is split evenly (17 vs. 17). 
We reject H0 at the 5% level, but this result is borderline and would not survive a correction for the number of tests run.

## On which groups drive the age and education results?

1. **Adjusted standardized residuals** per cell: values beyond about ±1.96 mean that cell deviates significantly from independence (at the 5% level).
2. **Pairwise 2x2 chi-square tests** between groups with a **Bonferroni correction** (the p-value is multiplied by the number of comparisons).

In [17]:
def adjusted_residuals(table):
    chi2, p, dof, expected = chi2_contingency(table, correction=False)
    n = table.values.sum()
    row = table.sum(axis=1).values.reshape(-1, 1) / n
    col = table.sum(axis=0).values.reshape(1, -1) / n
    adj = (table.values - expected) / np.sqrt(expected * (1 - row) * (1 - col))
    return pd.DataFrame(adj, index=table.index, columns=table.columns).round(2)

def pairwise_bonferroni(table):
    pairs = list(combinations(table.index, 2))
    rows = []
    for a, b in pairs:
        sub = table.loc[[a, b]]
        chi2, p, dof, _ = chi2_contingency(sub)
        rows.append({"group A": a, "group B": b, "chi2": round(chi2, 2),
                     "p": p, "p_bonferroni": min(p * len(pairs), 1.0)})
    return pd.DataFrame(rows).round(4)

from itertools import combinations

print("Age - adjusted standardized residuals")
print(adjusted_residuals(table))
print()
print("Age - pairwise tests (Bonferroni, 6 comparisons)")
print(pairwise_bonferroni(table))

Age - adjusted standardized residuals
party        Green   Red
age_group               
18-29         3.20 -3.20
30-44         0.43 -0.43
45-64        -1.65  1.65
65 or older  -1.19  1.19

Age - pairwise tests (Bonferroni, 6 comparisons)
  group A      group B  chi2       p  p_bonferroni
0   18-29        30-44  4.52  0.0335        0.2012
1   18-29        45-64  9.02  0.0027        0.0160
2   18-29  65 or older  8.12  0.0044        0.0263
3   30-44        45-64  0.74  0.3909        1.0000
4   30-44  65 or older  0.64  0.4252        1.0000
5   45-64  65 or older  0.00  1.0000        1.0000


In [18]:
print("Education - adjusted standardized residuals")
print(adjusted_residuals(table_edu))
print()
print("Education - pairwise tests (Bonferroni, 3 comparisons)")
print(pairwise_bonferroni(table_edu))

Education - adjusted standardized residuals
party      Green   Red
education             
High       -5.32  5.32
Low         1.94 -1.94
Medium      3.65 -3.65

Education - pairwise tests (Bonferroni, 3 comparisons)
  group A group B   chi2       p  p_bonferroni
0    High     Low  12.31  0.0004        0.0013
1    High  Medium  22.12  0.0000        0.0000
2     Low  Medium   0.00  1.0000        1.0000


**Age.** Only the 18-29 group deviates significantly from independence (adjusted residual +3.20 for Green, -3.20 for Red). Pairwise tests with Bonferroni correction confirm that 18-29 differs from 45-64 ($p_{adj} = 0.016$) and from 65 or older ($p_{adj} = 0.026$); the difference from 30-44 is not significant after correction ($p_{adj} = 0.201$). The 30-44, 45-64 and 65+ groups do not differ from each other. So the age effect is driven by the youngest voters being strongly Green.

**Education.** High education deviates most (residual -5.32 for Green, +5.32 for Red), Medium is more Green than expected (+3.65), and Low is borderline (+1.94). Pairwise, High differs significantly from both Low ($p_{adj} = 0.001$) and Medium ($p_{adj} < 0.001$), while Low and Medium are indistinguishable ($p_{adj} = 1.0$). So the education effect is that highly educated voters lean Red, compared with everyone else.
